In [ ]:
import andes
from andes.utils.paths import get_case

import matplotlib.pyplot as plt
# For Jupyter notebook
%matplotlib inline

In [ ]:
andes.config_logger(stream_level=40) # 10-DEBUG, 20-INFO, 30-WARNING, 40-ERROR, 50-CRITICAL

In [ ]:
ss = andes.run(get_case('ieee14/ieee14_fault.xlsx'), setup=False)
ss.GENROU.as_df()

In [ ]:
print(ss.TGOV1.pref0)
print(ss.PV.p)

In [ ]:
ss.TGOV1.as_df()

In [ ]:
# idxes = [4,5,7,9] # buses connected to induction (non-synchronous) generators
ss = andes.load(get_case('ieee14/ieee14_fault.xlsx'), setup=False)
# ss.Fault.alter('bus', 1, idx)  # arguments are `src`, `idx`, `value
# ss.Fault.alter('tc',  1, 1.2)  # arguments are `src`, `idx`, `value
# ss.add("Toggle", dict(model='SynGen', dev=ss.GENROU.idx.v[idx], t=1.0))
# ss.add("Toggle", dict(model='SynGen', dev=dict1['idx'], t=1.0))
ss.setup()
ss.PFlow.run()

ss.TDS.config.tf = 10
ss.TDS.run()
if ss.exit_code == 0:
    condition_initial = ss.TGOV1.pref0.v

In [ ]:
condition_initial

In [ ]:
# idxes = [4,5,7,9] # buses connected to induction (non-synchronous) generators
from typing import Any
import os

ss = andes.load(get_case('ieee14/ieee14_fault.xlsx'), setup=False)

results = dict[Any, Any]()
condition_record = dict[int, Any]()
for fault_idx in range(3):
    ref0_amount = [0.5, 0.7, 0.9][fault_idx]
    for idx in range(len(ss.TGOV1)): 
        ss = andes.load(get_case('ieee14/ieee14_fault.xlsx'), setup=False)
        ss.Fault.alter('u', 1, 0)
        # ss.Fault.alter('bus', 1, idx)  # arguments are `src`, `idx`, `value
        # ss.Fault.alter('tc',  1, 1.2)  # arguments are `src`, `idx`, `value
        ss.add('Alter', param_dict = {'name':ss.TGOV1.name.v[idx],  't':1.0, 'model':'TGOV1', 'dev':ss.TGOV1.idx.v[idx], 'src': 'pref0', 'method' :'*', 'amount':ref0_amount})
        # ss.add("Toggle", dict(model='SynGen', dev=ss.GENROU.idx.v[idx], t=1.0))
        # ss.add("Toggle", dict(model='SynGen', dev=dict1['idx'], t=1.0))
        ss.setup()
        ss.PFlow.run()

        ss.TDS.config.tf = 10
        ss.TDS.run()
        if ss.exit_code == 0:
            results[fault_idx*len(ss.TGOV1) + idx] = ss
            condition_record[fault_idx*len(ss.TGOV1) + idx] = ss.TGOV1.pref0.v

for ii in results.keys():
    results[ii].TDS.plt.export_csv(path=os.path.join('IEEE14_multi_traj','ieee14_reduce_tgov_out_'+str(ii)+'.csv'))
print(results.keys())

In [ ]:
condition_record

In [ ]:
import csv
import os
# Write condition_record to a CSV file
csv_filename = os.path.join('IEEE14_multi_traj','condition_record_wecc.csv')
with open(csv_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Header: first column is key, followed by 5 elements in value
    writer.writerow(['key', 'value1', 'value2', 'value3', 'value4', 'value5'])
    # Iterate over each item in condition_record
    for key, value_list in condition_record.items():
        # Ensure value_list has length 5; pad with empty strings if needed
        row = [key] + list(value_list) + [''] * max(0, 5 - len(value_list))
        writer.writerow(row)


In [ ]:
fig, ax = results[3].TDS.plt.plot((6, 7, 8, 9, 10), ylabel="Omega")

In [ ]:
fig, ax = results[4].TDS.plt.plot((6, 7, 8, 9, 10), ylabel="Omega")

In [ ]:
fig, ax = results[1].TDS.plt.plot((1, 2, 3, 4, 5), ylabel="delta")

In [ ]:
fig, ax = results[2].TDS.plt.plot((1, 2, 3, 4, 5), ylabel="delta")

In [ ]:
# print(ss.PQ.Req)
# print(ss.PQ.Xeq)
# print(ss.PQ.Ppf)
# print(ss.PQ.Qpf)
# print(ss.PQ.Ipeq)
# print(ss.PQ.Iqeq)
# print(ss.PQ.config.p2p)
# print(ss.PQ.config.p2i)
# print(ss.PQ.config.p2z)

In [ ]:
# idxes = [4,5,7,9] # buses connected to induction (non-synchronous) generators
ss = andes.load(get_case('ieee14/ieee14_fault.xlsx'), setup=False)

results = dict()

for idx in range(len(ss.PQ)):  
    ss = andes.load(get_case('ieee14/ieee14_fault.xlsx'), setup=False)
    ss.Fault.alter('u', 1, 0)
    # ss.Fault.alter('bus', 1, idx)  # arguments are `src`, `idx`, `value
    # ss.Fault.alter('tc',  1, 1.2)  # arguments are `src`, `idx`, `value
    ss.add('Alter', param_dict = {'name':ss.PQ.name.v[idx],  't':1.0, 'model':'PQ', 'dev':ss.PQ.idx.v[idx], 
        'src': 'Req', 'method' :'*', 'amount':2.0})
    ss.add('Alter', param_dict = {'name':ss.PQ.name.v[idx],  't':1.0, 'model':'PQ', 'dev':ss.PQ.idx.v[idx], 
        'src': 'Xeq', 'method' :'*', 'amount':2.0})    # ss.add("Toggle", dict(model='SynGen', dev=ss.GENROU.idx.v[idx], t=1.0))
    # ss.add("Toggle", dict(model='SynGen', dev=dict1['idx'], t=1.0))
    ss.setup()
    ss.PFlow.run()

    ss.TDS.config.tf = 10
    ss.TDS.run()
    if ss.exit_code == 0:
        results[idx] = ss
    
for ii in results.keys():
    results[ii].TDS.plt.export_csv(path=os.path.join('IEEE14_multi_traj','ieee14_increase_load_out_'+str(ii)+'.csv'))
print(results.keys())

In [ ]:
fig, ax = results[5].TDS.plt.plot((6, 7, 8, 9, 10), ylabel="Omega")

In [ ]:
fig, ax = results[2].TDS.plt.plot((1, 2, 3, 4, 5), ylabel="delta")

In [ ]:
results[1].TDS.plt.find('omega')

In [ ]:
results[2].TDS.plt.find('delta')